# 03 — Stage 1: Instrument Embedding (MIL + Attention)

Learn a **64-d song-level instrument embedding** with attention pooling over 15s windows.

**Bug-fix checklist (must hold here):**
1. `best_macro_map` updated inside the checkpoint-save branch
2. Val loader uses validation rows only (test never unioned in)
3. Mel paths resolve under `MEL_DIR` / `ROOT` only


## Kaggle setup (every notebook)

### A. Settings
1. Right sidebar → **Internet → On** (required for downloads).
2. **GPU**: Off for `00`/`01`/`04`–`06`. **GPU (T4)** on for `02`/`03`/`07`.

### B. How data moves (do not skip)
Kaggle **does not** keep `/kaggle/working` when you open a *new* notebook.

**After notebook 00 finishes:**
1. **Save Version** (top-right) → **Save & Run All** (or Quick Save if already finished).
2. Open **Advanced** → tick **Always save output**.
3. Wait until the version is **Success**.
4. Note the kernel slug (yours is **`thevifernando/dnn-download-data-1`**).

**In the next notebook (01, then 02, …):**
1. **Add Input** (right sidebar) → **Your notebooks** / **Notebook Output**.
2. Select **`dnn-download-data-1`** (latest successful version).
3. Files appear at `/kaggle/input/dnn-download-data-1/` (**read-only**).
4. This bootstrap **reads mels from that input** (does **not** copy 10 shards — they would overflow disk).
5. It **writes** new files (manifest, features, checkpoints) to `/kaggle/working/MTG_Instrument`.
6. **Save Version + save output** again so the *next* notebook can **Add Input** *this* notebook too (chain: 00 → 01 → 02 …).

### C. CLI (laptop only — not needed on Kaggle)
```bash
kaggle kernels output thevifernando/dnn-download-data-1 -p ./from_00
```
On Kaggle you **Add Input** instead of this command.

### D. GitHub
Commit **notebooks only** to `thevindu-branch`. Do **not** git-push the `.npy` shards (too large). Data stays on Kaggle output.


## Step 0 — Packages


In [ ]:
!pip install -q scikit-learn tqdm


## Step 1 — Bootstrap paths


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, time, urllib.request
import numpy as np
import pandas as pd

KERNEL_SLUG = "dnn-download-data-1"  # notebook 00 Kaggle slug — change if yours differs
WORKING_ROOT = Path("/kaggle/working/MTG_Instrument")
INPUT_BASE = Path("/kaggle/input")
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host: str = "github.com", port: int = 443, timeout: float = 5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def normalize_track_id(raw) -> str | None:
    """MTG ids are 7-digit zero-padded (track_0000948 → 0000948)."""
    m = re.search(r"(\d+)", str(raw))
    if not m:
        return None
    return f"{int(m.group(1)):07d}"


def _find_file(name: str, bases: list[Path]) -> Path | None:
    for base in bases:
        if not base.exists():
            continue
        hits = list(base.rglob(name))
        if hits:
            return hits[0]
    return None


def discover_input_root() -> Path | None:
    """Find a previous notebook-00 output or MTG data folder under /kaggle/input."""
    if not INPUT_BASE.exists():
        return None
    for marker in [
        "song_manifest.csv",
        "autotagging_genre-train.tsv",
        "autotagging_genre.tsv",
        ".shard_00_done",
    ]:
        hit = _find_file(marker, [INPUT_BASE])
        if hit is None:
            continue
        if marker == "song_manifest.csv":
            return hit.parents[1]  # .../MTG_Instrument/dataset/song_manifest.csv
        if marker == "autotagging_genre-train.tsv":
            # .../annotations/splits/split-0/file  OR  .../data/splits/split-0/file
            p = hit
            for _ in range(6):
                if (p / "dataset").exists() or p.name in {"MTG_Instrument", "data"}:
                    return p if p.name != "data" else p
                p = p.parent
            return hit.parents[2]
        if marker == "autotagging_genre.tsv":
            parent = hit.parent
            if parent.name == "annotations":
                return parent.parent
            return parent  # MTG data/
        if marker == ".shard_00_done":
            return hit.parents[2]  # .../MTG_Instrument/dataset/logmel_songs/.shard
    for p in INPUT_BASE.rglob("MTG_Instrument"):
        if p.is_dir():
            return p
    return None


def find_mel_dir() -> Path:
    """Prefer attached kernel output (read-only). Never copy 10 shards into working."""
    bases = [
        Path(f"/kaggle/input/{KERNEL_SLUG}") / "MTG_Instrument" / "dataset" / "logmel_songs",
        Path(f"/kaggle/input/{KERNEL_SLUG}") / "dataset" / "logmel_songs",
        WORKING_ROOT / "dataset" / "logmel_songs",
    ]
    kernel = Path(f"/kaggle/input/{KERNEL_SLUG}")
    extra = []
    if INPUT_BASE.exists():
        extra.append(INPUT_BASE)
    if kernel.exists():
        extra.append(kernel)
    for b in bases:
        if b.exists() and next(b.rglob("*.npy"), None) is not None:
            return b
    for b in extra:
        hit = next(b.rglob("*.npy"), None) if b.exists() else None
        if hit is None:
            continue
        p = hit.parent
        for _ in range(6):
            if p.name == "logmel_songs":
                return p
            p = p.parent
        return hit.parent
    return WORKING_ROOT / "dataset" / "logmel_songs"


def ensure_annotations(ann_dir: Path) -> Path:
    """Make sure split-0 TSVs exist; wget them if this is a fresh Kaggle session."""
    train = ann_dir / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if train.exists():
        return ann_dir

    # maybe files are flat, or under /kaggle/input with a different layout
    hit = _find_file("autotagging_genre-train.tsv", [ann_dir, INPUT_BASE, Path("/kaggle/working")])
    if hit is not None:
        dest = ann_dir / "splits" / "split-0" / hit.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if hit.resolve() != dest.resolve():
            shutil.copy2(hit, dest)
        # copy sibling split files from the same folder
        for name in [
            "autotagging_genre-validation.tsv",
            "autotagging_genre-test.tsv",
            "autotagging_instrument-train.tsv",
            "autotagging_instrument-validation.tsv",
            "autotagging_instrument-test.tsv",
        ]:
            sib = hit.parent / name
            if sib.exists():
                shutil.copy2(sib, dest.parent / name)
        genre_full = _find_file("autotagging_genre.tsv", [hit.parents[2] if len(hit.parents) > 2 else hit.parent, INPUT_BASE])
        if genre_full:
            shutil.copy2(genre_full, ann_dir / "autotagging_genre.tsv")
        inst_full = _find_file("autotagging_instrument.tsv", [hit.parents[2] if len(hit.parents) > 2 else hit.parent, INPUT_BASE])
        if inst_full:
            shutil.copy2(inst_full, ann_dir / "autotagging_instrument.tsv")
        print("Recovered split files from", hit.parent)
        return ann_dir

    if not check_internet():
        raise FileNotFoundError(
            "Split TSVs not found and Internet is OFF.\n"
            "Do ONE of:\n"
            "  A) Settings → Internet → On, re-run this cell (auto-download)\n"
            "  B) Add Data → attach notebook-00 output dataset (mtg-instrument-cache)\n"
            "  C) Stay in the SAME Kaggle session after running notebook 00"
        )

    print("Split TSVs missing — downloading official MTG annotations...")
    n = 0
    for rel in NEEDED_ANN:
        dest = ann_dir / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        url = f"{RAW_ANN}/{rel}"
        print("  wget", url)
        urllib.request.urlretrieve(url, dest)
        n += 1
    print(f"Downloaded {n} annotation files into {ann_dir}")
    return ann_dir


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    candidates = [
        ANN_DIR / "splits" / "split-0" / f"autotagging_{subset}-{split}.tsv",
        ANN_DIR / f"autotagging_{subset}-{split}.tsv",
        ANN_DIR / "splits" / "split-0" / f"{split}.tsv",
        ANN_DIR / f"{split}.tsv",
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        found = _find_file(f"autotagging_{subset}-{split}.tsv", [ANN_DIR, INPUT_BASE, Path("/kaggle/working")])
        path = found
    if path is None:
        raise FileNotFoundError(
            f"No split file for {subset}/{split}.\n"
            "Re-run the bootstrap cell after enabling Internet, or attach notebook-00 output."
        )
    df = pd.read_csv(path, sep="\t")
    col = "TRACK_ID" if "TRACK_ID" in df.columns else df.columns[0]
    ids = set()
    for v in df[col].astype(str):
        tid = normalize_track_id(v)
        if tid:
            ids.add(tid)
    print(f"{split:12s}  {len(ids):6d} ids   ← {path}")
    return ids


MEL_CACHE = Path("/kaggle/working/mel_cache")
MEL_CACHE.mkdir(parents=True, exist_ok=True)


def load_mel_npy(mel_abs, retries=5, pause=1.0):
    """Load mel with retries; cache under /kaggle/working for stable re-reads."""
    mel_abs = Path(mel_abs)
    sid = normalize_track_id(mel_abs.stem) or mel_abs.stem.replace("/", "_")
    cached = MEL_CACHE / f"{sid}.npy"
    if cached.exists():
        try:
            return np.load(cached)
        except (OSError, ValueError):
            cached.unlink(missing_ok=True)

    last_err = None
    for attempt in range(retries):
        try:
            arr = np.load(mel_abs, mmap_mode=None)
            arr = np.asarray(arr, dtype=np.float32)
            np.save(cached, arr)
            return arr
        except (OSError, ValueError) as e:
            last_err = e
            if attempt + 1 < retries:
                time.sleep(pause * (attempt + 1))
    nbytes = mel_abs.stat().st_size if mel_abs.exists() else "missing"
    raise RuntimeError(
        f"Bad/truncated mel — re-run notebook 00 for this shard: {mel_abs} "
        f"({nbytes} bytes). {last_err}"
    ) from last_err


def scan_bad_mels(df, label="manifest"):
    from tqdm.auto import tqdm

    bad = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"scan {label}"):
        try:
            load_mel_npy(row["mel_abs"])
        except Exception as e:
            bad.append({"song_id": str(row["song_id"]), "mel_abs": row["mel_abs"], "error": str(e)})
    if bad:
        out = RESULTS_DIR / f"bad_mels_{label}.json"
        out.write_text(json.dumps(bad, indent=2))
        print(f"WARNING: {len(bad)} bad mels → {out}")
    else:
        print(f"scan {label}: all {len(df)} mels OK (cache: {MEL_CACHE})")
    return bad


ONLINE = check_internet()
print("Internet reachable:", ONLINE)
print("KERNEL_SLUG =", KERNEL_SLUG)
print("/kaggle/input folders:", list(INPUT_BASE.iterdir()) if INPUT_BASE.exists() else "n/a")

ROOT = WORKING_ROOT
ROOT.mkdir(parents=True, exist_ok=True)
MEL_DIR = find_mel_dir()
ANN_DIR = ROOT / "annotations"
# if annotations only exist on the attached kernel, point there (read-only is OK)
for cand in [
    Path(f"/kaggle/input/{KERNEL_SLUG}") / "MTG_Instrument" / "annotations",
    Path(f"/kaggle/input/{KERNEL_SLUG}") / "annotations",
]:
    if (cand / "splits" / "split-0" / "autotagging_genre-train.tsv").exists():
        ANN_DIR = cand
        break
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"
att_manifest = _find_file("song_manifest.csv", [INPUT_BASE, Path("/kaggle/working")])
if not MANIFEST.exists() and att_manifest is not None:
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    try:
        shutil.copy2(att_manifest, MANIFEST)
        print("Copied song_manifest.csv from", att_manifest)
    except OSError:
        MANIFEST = att_manifest

for p in [ROOT / "dataset", ROOT / "annotations", FEAT_DIR, CKPT_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Small TSVs: copy/wget into working. Large mels stay on /kaggle/input.
ANN_DIR = ensure_annotations(ROOT / "annotations")

print("ROOT     =", ROOT)
print("MEL_DIR  =", MEL_DIR, "npy=", len(list(MEL_DIR.rglob('*.npy'))))
print("ANN_DIR  =", ANN_DIR)
print("split-0 train exists:", (ANN_DIR / "splits/split-0/autotagging_genre-train.tsv").exists())
print("MANIFEST =", MANIFEST, "exists=", MANIFEST.exists())


## Step 2 — Instrument multi-hot labels + manifest


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not MANIFEST.exists():
    raise FileNotFoundError("Run notebook 01 first.")
manifest = pd.read_csv(MANIFEST)
manifest["song_id"] = manifest["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)
EMBED_DIM, MAX_WINDOWS = 64, 12
N_MELS, N_FRAMES = 96, 1366
song_ids = manifest["song_id"].astype(str).tolist()

tag_to_idx, rows = {}, {s: set() for s in song_ids}
for path in [ANN_DIR / "autotagging_instrument.tsv", *ANN_DIR.rglob("*instrument*.tsv")]:
    if not Path(path).exists():
        continue
    df = pd.read_csv(path, sep="\t")
    id_col = "TRACK_ID" if "TRACK_ID" in df.columns else df.columns[0]
    tag_col = "TAGS" if "TAGS" in df.columns else df.columns[-1]
    for _, r in df.iterrows():
        sid = normalize_track_id(r[id_col])
        if sid not in rows:
            continue
        raw = r[tag_col]
        if pd.isna(raw):
            continue
        for tag in str(raw).replace("|", "\t").split("\t"):
            leaf = tag.strip().split("/")[-1].split("---")[-1]
            if not leaf or leaf.lower() in {"nan", "none", "tags"}:
                continue
            if leaf not in tag_to_idx:
                tag_to_idx[leaf] = len(tag_to_idx)
            rows[sid].add(leaf)
    if tag_to_idx:
        print("instruments from", path, "n=", len(tag_to_idx))
        break
assert tag_to_idx, "No instrument tags — re-run bootstrap / notebook 00"
INST_NAMES = [None] * len(tag_to_idx)
for t, i in tag_to_idx.items():
    INST_NAMES[i] = t
Y = np.zeros((len(song_ids), len(INST_NAMES)), np.float32)
id_to_idx = {s: i for i, s in enumerate(song_ids)}
for sid, tags in rows.items():
    i = id_to_idx[sid]
    for t in tags:
        Y[i, tag_to_idx[t]] = 1.0
print("Y_inst", Y.shape, "pos rate", float(Y.mean()))


## Step 3 — Window MIL dataset (path fallback stays under MEL_DIR)


In [ ]:
def resolve_stacked_mel_path(mel_abs: str) -> Path:
    p = Path(mel_abs)
    if p.exists():
        return p
    tid = normalize_track_id(p.stem)
    hits = list(MEL_DIR.rglob(f"*{tid}*.npy")) if tid else []
    if not hits:
        raise FileNotFoundError(f"mel not under MEL_DIR for {mel_abs}")
    return hits[0]


class WindowMILDataset(Dataset):
    def __init__(self, df, Y, id_to_idx, max_windows=MAX_WINDOWS, n_mels=N_MELS, n_frames=N_FRAMES):
        self.df = df.reset_index(drop=True)
        self.Y, self.id_to_idx = Y, id_to_idx
        self.max_windows, self.n_mels, self.n_frames = max_windows, n_mels, n_frames

    def __len__(self):
        return len(self.df)

    def _fix2d(self, x):
        """Force every window to exactly (n_mels, n_frames)."""
        x = np.asarray(x, dtype=np.float32)
        while x.ndim > 2:
            x = np.squeeze(x, axis=0)
        if x.ndim != 2:
            raise ValueError(f"expected 2D mel window, got {x.shape}")
        if x.shape[0] != self.n_mels and x.shape[1] == self.n_mels:
            x = x.T
        if x.shape[0] > self.n_mels:
            x = x[: self.n_mels]
        elif x.shape[0] < self.n_mels:
            x = np.pad(x, ((0, self.n_mels - x.shape[0]), (0, 0)))
        if x.shape[1] > self.n_frames:
            x = x[:, : self.n_frames]
        elif x.shape[1] < self.n_frames:
            x = np.pad(x, ((0, 0), (0, self.n_frames - x.shape[1])))
        if x.shape != (self.n_mels, self.n_frames):
            raise RuntimeError(f"mel fix failed: {x.shape}")
        return x

    def __getitem__(self, i):
        row = self.df.iloc[i]
        raw = load_mel_npy(resolve_stacked_mel_path(row["mel_abs"]))
        if raw.ndim == 2:
            raw = raw[None, ...]
        W = raw.shape[0]
        if W >= self.max_windows:
            windows, mask = raw[: self.max_windows], np.ones(self.max_windows, np.float32)
        else:
            windows = np.concatenate(
                [raw, np.zeros((self.max_windows - W, *raw.shape[1:]), raw.dtype)]
            )
            mask = np.array([1] * W + [0] * (self.max_windows - W), np.float32)
        out = np.zeros((self.max_windows, self.n_mels, self.n_frames), np.float32)
        for j, w in enumerate(windows):
            out[j] = self._fix2d(w)
        y = self.Y[self.id_to_idx[str(row["song_id"])]]
        return (
            torch.from_numpy(out[:, None, :, :]),
            torch.from_numpy(mask),
            torch.from_numpy(y),
            str(row["song_id"]),
        )


def make_loader(split, bs=8, shuffle=False):
    sub = manifest[manifest["split"] == split]
    assert set(sub["split"].unique()) == {split}
    return DataLoader(WindowMILDataset(sub, Y, id_to_idx), batch_size=bs, shuffle=shuffle, num_workers=0)


## Step 4 — Encoder + attention pool + instrument head


In [ ]:
class WindowEncoder(nn.Module):
    def __init__(self, emb=EMBED_DIM):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.proj = nn.Linear(64, emb)

    def forward(self, x):
        B, W, C, M, T = x.shape
        h = self.cnn(x.reshape(B * W, C, M, T)).flatten(1)
        return self.proj(h).reshape(B, W, -1)


class AttnPool(nn.Module):
    def __init__(self, emb=EMBED_DIM):
        super().__init__()
        self.score = nn.Linear(emb, 1)

    def forward(self, H, mask):
        logits = self.score(H).squeeze(-1).masked_fill(mask < 0.5, -1e9)
        w = torch.softmax(logits, dim=-1)
        z = torch.sum(H * w.unsqueeze(-1), dim=1)
        return z, w


class Stage1Model(nn.Module):
    def __init__(self, n_tags, emb=EMBED_DIM):
        super().__init__()
        self.enc = WindowEncoder(emb)
        self.pool = AttnPool(emb)
        self.head = nn.Linear(emb, n_tags)

    def forward(self, x, mask):
        H = self.enc(x)
        z, attn = self.pool(H, mask)
        return self.head(z), z, attn

model = Stage1Model(n_tags=Y.shape[1]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()


## Step 5 — Train + export 64-d embeddings for every manifest song


In [ ]:
def macro_map(y_true, y_prob):
    scores = []
    for k in range(y_true.shape[1]):
        if y_true[:, k].sum() in (0, len(y_true)):
            continue
        try:
            scores.append(average_precision_score(y_true[:, k], y_prob[:, k]))
        except ValueError:
            continue
    return float(np.mean(scores)) if scores else float("nan")


@torch.no_grad()
def eval_split(loader):
    model.eval()
    ys, ps = [], []
    for x, mask, y, _ in loader:
        logits, _, _ = model(x.to(DEVICE), mask.to(DEVICE))
        ps.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(y.numpy())
    return macro_map(np.concatenate(ys), np.concatenate(ps))

train_loader, val_loader, test_loader = make_loader("train", shuffle=True), make_loader("validation"), make_loader("test")
_x, _m, _y, _ = next(iter(train_loader))
print("preflight batch", tuple(_x.shape), "expect (bs, 12, 1, 96, 1366)")
assert _x.shape[2:] == (1, N_MELS, N_FRAMES), f"re-run this entire cell — got {_x.shape}"
SCAN_MELS = False
if SCAN_MELS:
    bad = scan_bad_mels(manifest, "all")
    if bad:
        raise RuntimeError(f"{len(bad)} bad mels — see {RESULTS_DIR}/bad_mels_all.json")
EPOCHS = 8
best_macro_map = 0.0
ckpt_dir = CKPT_DIR / "stage1"
ckpt_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for x, mask, y, _ in tqdm(train_loader, leave=False):
        x, mask, y = x.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        logits, _, _ = model(x, mask)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()
        total += loss.item() * len(x)
    val_map = eval_split(val_loader)
    print(f"epoch {epoch}: loss={total/len(train_loader.dataset):.4f} val_macro_map={val_map:.4f}")
    if val_map > best_macro_map:
        best_macro_map = val_map
        torch.save({"model": model.state_dict(), "best_macro_map": best_macro_map, "epoch": epoch, "tags": INST_NAMES},
                   ckpt_dir / "best.pt")
        print("  ✓ checkpoint", best_macro_map)

state = torch.load(ckpt_dir / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model"])
model.eval()
all_loader = DataLoader(WindowMILDataset(manifest, Y, id_to_idx), batch_size=8, num_workers=0)
embeds, ids = [], []
with torch.no_grad():
    for x, mask, y, sid in tqdm(all_loader):
        _, z, _ = model(x.to(DEVICE), mask.to(DEVICE))
        embeds.append(z.cpu().numpy())
        ids.extend(list(sid))
E = np.concatenate(embeds, 0)
out = FEAT_DIR / "instrument"
out.mkdir(parents=True, exist_ok=True)
np.save(out / "instrument_embeddings.npy", E)
(out / "song_ids.json").write_text(json.dumps(ids))
print("saved", E.shape, "test macro_map", eval_split(test_loader))
